# LSTM Dataset Preparation

This notebook prepares continuous time-series datasets (TST) for LSTM training: it loads quarterly data, filters the time range, creates numeric country codes, defines indicators, creates continuous sequences, and saves PyTorch tensors.


In [26]:
import os
import pandas as pd
import numpy as np

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4"


In [27]:
df_weo = pd.read_csv("quarterly/df_res_light_data_sms.csv", index_col=0)
df_weo.index = [x for x in range(len(df_weo))]
df_weo


,date,country,GDP,出口金额,工业,股票市值,国际收支金融账户差额,国际收支经常账户差额,国际收支经常账户贷方,国际收支经常账户借方,...,零售额,CPI,失业率,中央银行政策利率,quarter,ADM0_ISO,sum,mean,std,year
0,2013-03-31,中国,7.9,18.903333,9.600000,4.870704,82.968290,43.440651,13.953076,12.280781,...,12.450100,2.439967,4.100000,6.000000,1,CHN,2.233006e+07,0.428795,2.806715,2013
1,2013-06-30,中国,7.6,4.130000,9.133333,-2.221363,38.479229,-27.681847,6.181969,9.874452,...,13.001567,2.384200,4.100000,6.000000,2,CHN,1.772255e+07,0.328655,2.572944,2013
2,2013-09-30,中国,7.9,3.920000,10.100000,6.091360,-61.013307,-57.231530,5.003561,13.519923,...,13.298900,2.764200,4.040000,6.000000,3,CHN,1.819013e+07,0.342631,3.591020,2013
3,2013-12-31,中国,7.7,7.466667,10.000000,7.460061,-131.056027,-38.077580,8.980957,13.196985,...,13.547167,2.907500,4.050000,6.000000,4,CHN,2.870266e+07,0.532995,3.410764,2013
4,2014-03-31,中国,7.5,-4.743333,8.700000,-8.635409,-24.056782,-86.634857,-1.796143,4.350455,...,12.032500,2.274000,4.080000,6.000000,1,CHN,2.299434e+07,0.435236,3.457568,2014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549,2021-06-30,波兰,12.2,58.923333,31.166667,63.838796,-73.177749,-51.692650,49.850488,56.750866,...,12.163333,4.354821,3.566667,0.100000,2,POL,1.990839e+05,0.059004,0.825555,2021
550,2021-09-30,波兰,7.2,17.103333,10.433333,50.821247,-169.547694,-419.192964,16.362307,26.194703,...,8.673333,5.268148,3.200000,0.100000,3,POL,6.628967e+05,0.196482,1.104317,2021
551,2021-12-31,波兰,9.3,11.770000,13.000000,28.693169,-141.769342,-172.908864,9.560565,23.349057,...,12.030000,7.781450,3.033333,1.166667,4,POL,4.031915e+06,1.194956,5.594065,2021
552,2022-03-31,波兰,10.5,11.183333,16.200000,5.301908,-1405.304010,-945.956873,12.907786,20.145678,...,12.533333,9.637951,2.733333,2.833333,1,POL,4.144025e+06,1.228150,5.478767,2022


In [28]:
df_weo.columns


Index(['date', 'country', 'GDP', '出口金额', '工业', '股票市值', '国际收支金融账户差额',
       '国际收支经常账户差额', '国际收支经常账户贷方', '国际收支经常账户借方', '国际收支资本账户差额', '国际收支资本账户贷方',
       '国际收支资本账户借方', '国际收支差额', '国际投资头寸资产', '国际投资头寸负债', '国际投资头寸净额', '进口金额',
       '名义有效汇率', '零售额', 'CPI', '失业率', '中央银行政策利率', 'quarter', 'ADM0_ISO', 'sum',
       'mean', 'std', 'year'],
      dtype='object')

In [29]:
df_weo = df_weo.dropna(subset=["GDP"])
df_weo.index = [x for x in range(len(df_weo))]


In [30]:
df_weo


,date,country,GDP,出口金额,工业,股票市值,国际收支金融账户差额,国际收支经常账户差额,国际收支经常账户贷方,国际收支经常账户借方,...,零售额,CPI,失业率,中央银行政策利率,quarter,ADM0_ISO,sum,mean,std,year
0,2013-03-31,中国,7.9,18.903333,9.600000,4.870704,82.968290,43.440651,13.953076,12.280781,...,12.450100,2.439967,4.100000,6.000000,1,CHN,2.233006e+07,0.428795,2.806715,2013
1,2013-06-30,中国,7.6,4.130000,9.133333,-2.221363,38.479229,-27.681847,6.181969,9.874452,...,13.001567,2.384200,4.100000,6.000000,2,CHN,1.772255e+07,0.328655,2.572944,2013
2,2013-09-30,中国,7.9,3.920000,10.100000,6.091360,-61.013307,-57.231530,5.003561,13.519923,...,13.298900,2.764200,4.040000,6.000000,3,CHN,1.819013e+07,0.342631,3.591020,2013
3,2013-12-31,中国,7.7,7.466667,10.000000,7.460061,-131.056027,-38.077580,8.980957,13.196985,...,13.547167,2.907500,4.050000,6.000000,4,CHN,2.870266e+07,0.532995,3.410764,2013
4,2014-03-31,中国,7.5,-4.743333,8.700000,-8.635409,-24.056782,-86.634857,-1.796143,4.350455,...,12.032500,2.274000,4.080000,6.000000,1,CHN,2.299434e+07,0.435236,3.457568,2014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
549,2021-06-30,波兰,12.2,58.923333,31.166667,63.838796,-73.177749,-51.692650,49.850488,56.750866,...,12.163333,4.354821,3.566667,0.100000,2,POL,1.990839e+05,0.059004,0.825555,2021
550,2021-09-30,波兰,7.2,17.103333,10.433333,50.821247,-169.547694,-419.192964,16.362307,26.194703,...,8.673333,5.268148,3.200000,0.100000,3,POL,6.628967e+05,0.196482,1.104317,2021
551,2021-12-31,波兰,9.3,11.770000,13.000000,28.693169,-141.769342,-172.908864,9.560565,23.349057,...,12.030000,7.781450,3.033333,1.166667,4,POL,4.031915e+06,1.194956,5.594065,2021
552,2022-03-31,波兰,10.5,11.183333,16.200000,5.301908,-1405.304010,-945.956873,12.907786,20.145678,...,12.533333,9.637951,2.733333,2.833333,1,POL,4.144025e+06,1.228150,5.478767,2022


In [31]:
df_weo["year"].unique().min()


np.int64(2013)

In [32]:
df_weo["year"].unique().max()


np.int64(2022)

## Select Time Period

Filter the dataset to the analysis period (2013-2019).


In [33]:
df_weo = df_weo.loc[(df_weo["year"] >= 2013) & (df_weo["year"] <= 2019)]
# df_weo


In [34]:
df_filter = df_weo.dropna()

# Rename columns from Chinese to English
indicator_name_map = {
    "出口金额": "export_value",
    "工业": "industry",
    "股票市值": "market_cap",
    "国际收支金融账户差额": "balance_financial_account",
    "国际收支经常账户差额": "current_account_balance",
    "国际收支经常账户贷方": "current_account_credit",
    "国际收支经常账户借方": "current_account_debit",
    "国际收支资本账户差额": "capital_account_balance",
    "国际收支资本账户贷方": "capital_account_credit",
    "国际收支资本账户借方": "capital_account_debit",
    "国际收支差额": "balance_of_payments",
    "国际投资头寸资产": "international_investment_position_assets",
    "国际投资头寸负债": "international_investment_position_liabilities",
    "国际投资头寸净额": "international_investment_position_net",
    "进口金额": "import_value",
    "名义有效汇率": "neer",
    "零售额": "retail_sales",
    "CPI": "cpi",
    "失业率": "unemployment_rate",
    "中央银行政策利率": "policy_rate",
}

# Rename columns that exist in df_filter
rename_dict = {chinese: english for chinese, english in indicator_name_map.items() if chinese in df_filter.columns}
df_filter = df_filter.rename(columns=rename_dict)

print("Columns renamed to English. Available columns:", df_filter.columns.tolist())
df_filter


Columns renamed to English. Available columns: ['date', 'country', 'GDP', 'export_value', 'industry', 'market_cap', 'balance_financial_account', 'current_account_balance', 'current_account_credit', 'current_account_debit', 'capital_account_balance', 'capital_account_credit', 'capital_account_debit', 'balance_of_payments', 'international_investment_position_assets', 'international_investment_position_liabilities', 'international_investment_position_net', 'import_value', 'neer', 'retail_sales', 'cpi', 'unemployment_rate', 'policy_rate', 'quarter', 'ADM0_ISO', 'sum', 'mean', 'std', 'year']


,date,country,GDP,export_value,industry,market_cap,balance_financial_account,current_account_balance,current_account_credit,current_account_debit,...,retail_sales,cpi,unemployment_rate,policy_rate,quarter,ADM0_ISO,sum,mean,std,year
0,2013-03-31,中国,7.9,18.903333,9.600000,4.870704,82.968290,43.440651,13.953076,12.280781,...,12.450100,2.439967,4.100000,6.0,1,CHN,2.233006e+07,0.428795,2.806715,2013
1,2013-06-30,中国,7.6,4.130000,9.133333,-2.221363,38.479229,-27.681847,6.181969,9.874452,...,13.001567,2.384200,4.100000,6.0,2,CHN,1.772255e+07,0.328655,2.572944,2013
2,2013-09-30,中国,7.9,3.920000,10.100000,6.091360,-61.013307,-57.231530,5.003561,13.519923,...,13.298900,2.764200,4.040000,6.0,3,CHN,1.819013e+07,0.342631,3.591020,2013
3,2013-12-31,中国,7.7,7.466667,10.000000,7.460061,-131.056027,-38.077580,8.980957,13.196985,...,13.547167,2.907500,4.050000,6.0,4,CHN,2.870266e+07,0.532995,3.410764,2013
4,2014-03-31,中国,7.5,-4.743333,8.700000,-8.635409,-24.056782,-86.634857,-1.796143,4.350455,...,12.032500,2.274000,4.080000,6.0,1,CHN,2.299434e+07,0.435236,3.457568,2014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
539,2018-12-31,波兰,5.5,4.573333,4.533333,-19.726309,118.071429,-16.388226,4.779318,4.953365,...,5.456667,1.513641,3.900000,1.5,4,POL,3.202875e+06,1.351579,8.123920,2018
540,2019-03-31,波兰,5.1,0.613333,6.766667,-17.303037,-27.366864,49.811637,0.300426,-1.087284,...,4.656667,1.189471,3.800000,1.5,1,POL,4.236406e+06,1.787682,8.045791,2019
541,2019-06-30,波兰,4.6,0.023333,4.600000,-5.605810,21.984127,37.079163,0.954344,-0.016254,...,4.773333,2.365209,3.333333,1.5,2,POL,5.154828e+05,0.219359,1.906101,2019
542,2019-09-30,波兰,4.1,1.756667,2.566667,-13.293187,166.681594,57.431158,2.790755,-0.922125,...,4.426667,2.681516,3.200000,1.5,3,POL,1.006961e+06,0.425211,2.717304,2019


In [35]:
# English indicator list (replaces the original Chinese `data_v` list)
# Keep names short and clear for notebook readability; actual df column mapping is handled below
data_v = [
    "export_value",
    "industry",
    "market_cap",
    "balance_financial_account",
    "current_account_balance",
    "current_account_credit",
    "current_account_debit",
    "capital_account_balance",
    "capital_account_credit",
    "capital_account_debit",
    "balance_of_payments",
    "international_investment_position_assets",
    "international_investment_position_liabilities",
    "international_investment_position_net",
    "import_value",
    "neer",
    "retail_sales",
    "cpi",
    "unemployment_rate",
    "policy_rate",
    "sum",
    "mean",
    "std",
    "GDP",
]
print(
    "Declared English feature list in variable `data_v` (will be mapped to actual df columns)"
)


Declared English feature list in variable `data_v` (will be mapped to actual df columns)


In [36]:
label_v = ["GDP"]


In [37]:
# Extract features using English column names (columns are now renamed)
data = df_filter[data_v]
print("Feature data shape:", data.shape)


Feature data shape: (413, 24)


In [38]:
data.shape


(413, 24)

In [39]:
df_filter


,date,country,GDP,export_value,industry,market_cap,balance_financial_account,current_account_balance,current_account_credit,current_account_debit,...,retail_sales,cpi,unemployment_rate,policy_rate,quarter,ADM0_ISO,sum,mean,std,year
0,2013-03-31,中国,7.9,18.903333,9.600000,4.870704,82.968290,43.440651,13.953076,12.280781,...,12.450100,2.439967,4.100000,6.0,1,CHN,2.233006e+07,0.428795,2.806715,2013
1,2013-06-30,中国,7.6,4.130000,9.133333,-2.221363,38.479229,-27.681847,6.181969,9.874452,...,13.001567,2.384200,4.100000,6.0,2,CHN,1.772255e+07,0.328655,2.572944,2013
2,2013-09-30,中国,7.9,3.920000,10.100000,6.091360,-61.013307,-57.231530,5.003561,13.519923,...,13.298900,2.764200,4.040000,6.0,3,CHN,1.819013e+07,0.342631,3.591020,2013
3,2013-12-31,中国,7.7,7.466667,10.000000,7.460061,-131.056027,-38.077580,8.980957,13.196985,...,13.547167,2.907500,4.050000,6.0,4,CHN,2.870266e+07,0.532995,3.410764,2013
4,2014-03-31,中国,7.5,-4.743333,8.700000,-8.635409,-24.056782,-86.634857,-1.796143,4.350455,...,12.032500,2.274000,4.080000,6.0,1,CHN,2.299434e+07,0.435236,3.457568,2014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
539,2018-12-31,波兰,5.5,4.573333,4.533333,-19.726309,118.071429,-16.388226,4.779318,4.953365,...,5.456667,1.513641,3.900000,1.5,4,POL,3.202875e+06,1.351579,8.123920,2018
540,2019-03-31,波兰,5.1,0.613333,6.766667,-17.303037,-27.366864,49.811637,0.300426,-1.087284,...,4.656667,1.189471,3.800000,1.5,1,POL,4.236406e+06,1.787682,8.045791,2019
541,2019-06-30,波兰,4.6,0.023333,4.600000,-5.605810,21.984127,37.079163,0.954344,-0.016254,...,4.773333,2.365209,3.333333,1.5,2,POL,5.154828e+05,0.219359,1.906101,2019
542,2019-09-30,波兰,4.1,1.756667,2.566667,-13.293187,166.681594,57.431158,2.790755,-0.922125,...,4.426667,2.681516,3.200000,1.5,3,POL,1.006961e+06,0.425211,2.717304,2019


In [40]:
df_filter["GDP"].min()


np.float64(-4.2628)

In [41]:
df_filter["GDP"].max()


np.float64(11.1)

## Create TST


In [42]:
def create_continuous_time_series_data(
    df, country_col, year_col, quarter_col, indicator_cols, time_length
):
    """Create continuous time-series sliding windows for each country.

    Args:
        df (DataFrame): input dataframe containing country, year, quarter and indicators.
        country_col (str): name of country column.
        year_col (str): name of year column.
        quarter_col (str): name of quarter column.
        indicator_cols (list): list of column names to include in each sequence (must exist in df).
        time_length (int): number of time steps in each input sequence.

    Returns:
        sequences (np.ndarray): shape (n_samples, time_length, n_features)
        targets (np.ndarray): shape (n_samples, n_features) - target at next time step
    """
    # sort by country, year and quarter
    df = df.sort_values(by=[country_col, year_col, quarter_col])

    # storage
    sequences = []
    targets = []

    # list of countries
    countries = df[country_col].unique()

    # iterate countries
    for country in countries:
        country_data = df[df[country_col] == country]

        years = country_data[year_col].values
        quarters = country_data[quarter_col].values

        # sliding windows
        for start_idx in range(len(years) - time_length):
            end_idx = start_idx + time_length
            # check year/quarter continuity (each step increases by 1 quarter)
            if np.all(
                (
                    np.diff(years[start_idx:end_idx]) * 4
                    + np.diff(quarters[start_idx:end_idx])
                )
                == 1
            ):
                sequence = country_data.iloc[start_idx:end_idx][indicator_cols].values
                target = country_data.iloc[end_idx : end_idx + 1][indicator_cols].values
                sequences.append(sequence)
                targets.append(target[-1])

    return np.array(sequences), np.array(targets)


## Create Continuous Time-Series (TST)

This section builds sliding windows (sequences) per country for LSTM training.


In [43]:
country_list = df_filter["country"].unique()


In [44]:
code_map = {}
for i in range(len(country_list)):
    code_map[country_list[i]] = i


In [45]:
df_filter["country_code"] = df_filter["country"].apply(lambda x: code_map[x])


In [46]:
df_filter


,date,country,GDP,export_value,industry,market_cap,balance_financial_account,current_account_balance,current_account_credit,current_account_debit,...,cpi,unemployment_rate,policy_rate,quarter,ADM0_ISO,sum,mean,std,year,country_code
0,2013-03-31,中国,7.9,18.903333,9.600000,4.870704,82.968290,43.440651,13.953076,12.280781,...,2.439967,4.100000,6.0,1,CHN,2.233006e+07,0.428795,2.806715,2013,0
1,2013-06-30,中国,7.6,4.130000,9.133333,-2.221363,38.479229,-27.681847,6.181969,9.874452,...,2.384200,4.100000,6.0,2,CHN,1.772255e+07,0.328655,2.572944,2013,0
2,2013-09-30,中国,7.9,3.920000,10.100000,6.091360,-61.013307,-57.231530,5.003561,13.519923,...,2.764200,4.040000,6.0,3,CHN,1.819013e+07,0.342631,3.591020,2013,0
3,2013-12-31,中国,7.7,7.466667,10.000000,7.460061,-131.056027,-38.077580,8.980957,13.196985,...,2.907500,4.050000,6.0,4,CHN,2.870266e+07,0.532995,3.410764,2013,0
4,2014-03-31,中国,7.5,-4.743333,8.700000,-8.635409,-24.056782,-86.634857,-1.796143,4.350455,...,2.274000,4.080000,6.0,1,CHN,2.299434e+07,0.435236,3.457568,2014,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
539,2018-12-31,波兰,5.5,4.573333,4.533333,-19.726309,118.071429,-16.388226,4.779318,4.953365,...,1.513641,3.900000,1.5,4,POL,3.202875e+06,1.351579,8.123920,2018,16
540,2019-03-31,波兰,5.1,0.613333,6.766667,-17.303037,-27.366864,49.811637,0.300426,-1.087284,...,1.189471,3.800000,1.5,1,POL,4.236406e+06,1.787682,8.045791,2019,16
541,2019-06-30,波兰,4.6,0.023333,4.600000,-5.605810,21.984127,37.079163,0.954344,-0.016254,...,2.365209,3.333333,1.5,2,POL,5.154828e+05,0.219359,1.906101,2019,16
542,2019-09-30,波兰,4.1,1.756667,2.566667,-13.293187,166.681594,57.431158,2.790755,-0.922125,...,2.681516,3.200000,1.5,3,POL,1.006961e+06,0.425211,2.717304,2019,16


In [47]:
len(df_filter["country"].unique())


17

In [48]:
# Define indicator columns using English names (columns are now already renamed to English)
indicator_cols_english = [
    "export_value",
    "industry",
    "market_cap",
    "balance_financial_account",
    "current_account_balance",
    "current_account_credit",
    "current_account_debit",
    "capital_account_balance",
    "capital_account_credit",
    "capital_account_debit",
    "balance_of_payments",
    "international_investment_position_assets",
    "international_investment_position_liabilities",
    "international_investment_position_net",
    "import_value",
    "neer",
    "retail_sales",
    "cpi",
    "unemployment_rate",
    "policy_rate",
    "sum",
    "mean",
    "std",
]

# Build the indicator columns list from English names (columns are already renamed)
indicator_cols = []
missing = []
for eng in indicator_cols_english:
    if eng in df_filter.columns:
        indicator_cols.append(eng)
    else:
        missing.append(eng)

if missing:
    missing_names = ", ".join(missing)
    raise KeyError(f"Indicator columns not found in dataframe: {missing_names}")

# Append GDP and meta columns if present
if "GDP" in df_filter.columns:
    label_v = ["GDP"]
else:
    label_v = []

# Add meta columns if they exist
if "country_code" in df_filter.columns:
    indicator_cols.append("country_code")
if "year" in df_filter.columns:
    indicator_cols.append("year")
if "quarter" in df_filter.columns:
    indicator_cols.append("quarter")

# Show resolved indicator columns
print("Using indicator columns:", indicator_cols)


Using indicator columns: ['export_value', 'industry', 'market_cap', 'balance_financial_account', 'current_account_balance', 'current_account_credit', 'current_account_debit', 'capital_account_balance', 'capital_account_credit', 'capital_account_debit', 'balance_of_payments', 'international_investment_position_assets', 'international_investment_position_liabilities', 'international_investment_position_net', 'import_value', 'neer', 'retail_sales', 'cpi', 'unemployment_rate', 'policy_rate', 'sum', 'mean', 'std', 'country_code', 'year', 'quarter']


In [49]:
import torch

# Define column names (now using English names)
country_col = "country"
year_col = "year"
quarter_col = "quarter"


def save_data_label():
    for time_length in [8, 10, 12]:
        sequences, targets = create_continuous_time_series_data(
            df_filter, country_col, year_col, quarter_col, indicator_cols, time_length
        )
        print(sequences.shape)
        print(targets.shape)

        # for
        data = torch.Tensor(sequences)
        data.size()

        torch.save(
            data, "../dataset/LSTM_data_light_sms_q_t" + str(time_length) + "_13-19.pt"
        )

        labels = torch.Tensor(targets)
        labels.size()

        torch.save(
            labels,
            "../dataset/LSTM_label_light_sms_q_t" + str(time_length) + "_13-19.pt",
        )

        flattened_data = data.view(data.size()[0], -1)
        flattened_data.size()

        torch.save(
            flattened_data,
            "../dataset/LSTM_flattened_data_light_sms_q_t"
            + str(time_length)
            + "_13-19.pt",
        )

        print(flattened_data.size())


In [50]:
save_data_label()


(258, 8, 26)
(258, 26)
torch.Size([258, 208])
(226, 10, 26)
(226, 26)
torch.Size([226, 260])
(195, 12, 26)
(195, 26)
torch.Size([195, 312])
